# Estimativa de Movimento de Câmera FLIR — Rastreamento de UAP

## Referência visual: características do vídeo FLIR de sistema de armas

Baseado na análise visual de frames típicos desse tipo de vídeo, o código lida com:

| Elemento | Aparência | Tratamento |
|---|---|---|
| **Áreas pretas irregulares** | Blocos pretos nos cantos/bordas — overlay do sistema, fora do campo da câmera | Mascaradas automaticamente por limiar de intensidade + morfologia |
| **Alvo UAP** | Blob branco intenso (white-hot) ou escuro (black-hot) próximo ao centro | Detectado adaptativamente por percentil e mascarado com dilatação |
| **Mira / crosshair** | Símbolo + no centro da imagem, fixo | Banda horizontal + vertical + caixa central mascaradas |
| **Texto HUD** | Letras como 'N' (bússola), números de altitude, etc. | Mascaráveis via `hud_boxes` ou por detecção de texto branco |
| **Background** | Nuvens / terreno com textura moderada | Único elemento usado para estimativa de movimento |

## Abordagem

```
Frame FLIR
   │
   ├─► _find_valid_region()     detecta área não-preta (campo real da câmera)
   ├─► _detect_polarity()       identifica white-hot vs black-hot automaticamente
   ├─► _mask_target_blob()      remove UAP (blob de alto contraste)
   ├─► _mask_reticle()          remove mira/crosshair central
   └─► máscara final de background
           │
           ▼
   Shi-Tomasi corners → Lucas-Kanade pyramidal → vetores de deslocamento
           │
           ▼
   Filtragem MAD (rejeita outliers) → mediana robusta → (bg_dx, bg_dy)
           │
           ▼
   Pré-filtro mediana móvel → soma cumulativa → posição câmera
           │
           ▼
   Savitzky-Golay (deriv=0,1,2) → posição suave, velocidade, aceleração
```

## Convenção de sinal
- `cam_pos_x > 0` → câmera girou para a **direita** (pan direita)  
- `cam_pos_y > 0` → câmera inclinou para **baixo** (coords de imagem: y cresce para baixo)  
- `bg_dx_px` = deslocamento do **fundo** (sinal inverso à câmera)

## Dependências

```bash
pip install opencv-python numpy pandas matplotlib scipy
```

In [ ]:
from __future__ import annotations

import math
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

try:
    from scipy.signal import savgol_filter
    _SCIPY = True
except ImportError:
    _SCIPY = False
    warnings.warn("scipy não encontrado — instale com: pip install scipy\n"
                  "Usando numpy.gradient como fallback (resultado mais ruidoso).")

try:
    import cv2
except ImportError as _err:
    raise ImportError("OpenCV é necessário: pip install opencv-python") from _err

print(f"OK  OpenCV {cv2.__version__}")
print(f"OK  scipy={'disponível' if _SCIPY else 'AUSENTE — instale para melhores resultados'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  BLOCO 1 — PRÉ-PROCESSAMENTO E MÁSCARA AUTOMÁTICA
# ═══════════════════════════════════════════════════════════════════════════════

def _to_gray(frame: np.ndarray) -> np.ndarray:
    """
    Converte frame BGR (ou já cinza) para uint8 single-channel.
    Aplica blur 3×3 suave para reduzir ruído de compressão de vídeo
    sem destruir as bordas de textura usadas no rastreamento.
    """
    if frame.ndim == 3:
        g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        g = frame.copy()
    return cv2.GaussianBlur(g, (3, 3), 0)


def _find_valid_region(
    gray: np.ndarray,
    black_thresh: int = 18,
    close_px: int = 25,
    erode_px: int = 10,
) -> np.ndarray:
    """
    Detecta a região visível (não-preta) do frame FLIR.

    Em vídeos de targeting pod e câmeras de sistemas de armas, o campo de visão
    da câmera NÃO preenche o frame inteiro. As bordas/cantos são preenchidas com
    blocos pretos irregulares pelo pipeline de renderização da HUD. Esses blocos
    têm intensidade ≈ 0 e precisam ser removidos ANTES de estimar o movimento.

    Algoritmo:
      1. Limiar simples: pixels > black_thresh são candidatos a válidos
      2. Fechamento morfológico: une regiões vizinhas separadas por costuras
      3. Erosão: afasta a máscara das bordas dos blocos pretos (evita artefatos)
    """
    valid = (gray > black_thresh).astype(np.uint8) * 255
    k_close = cv2.getStructuringElement(cv2.MORPH_RECT, (close_px, close_px))
    valid = cv2.morphologyEx(valid, cv2.MORPH_CLOSE, k_close)
    k_erode = cv2.getStructuringElement(cv2.MORPH_RECT, (erode_px, erode_px))
    valid = cv2.erode(valid, k_erode)
    return valid


def _detect_thermal_polarity(
    gray: np.ndarray,
    valid_region: np.ndarray,
    center_frac: float = 0.15,
) -> str:
    """
    Detecta automaticamente se a câmera FLIR está em modo white-hot ou black-hot.

    Heurística: compara a intensidade média da região central (onde o alvo UAP
    rastreado pelo sistema de armas tende a ficar) com a média do resto do frame.
    Se o centro for mais brilhante que o fundo → white-hot; caso contrário → black-hot.

    Retorna 'white_hot' ou 'black_hot'.
    """
    h, w = gray.shape[:2]
    cy, cx = h // 2, w // 2
    ch = max(1, int(h * center_frac / 2))
    cw = max(1, int(w * center_frac / 2))

    center_mask = np.zeros((h, w), dtype=np.uint8)
    center_mask[cy - ch : cy + ch, cx - cw : cx + cw] = 255
    center_mask = cv2.bitwise_and(center_mask, valid_region)
    surround_mask = cv2.bitwise_and(valid_region, cv2.bitwise_not(center_mask))

    center_px   = gray[center_mask > 0]
    surround_px = gray[surround_mask > 0]

    if center_px.size < 10 or surround_px.size < 10:
        return "white_hot"
    return "white_hot" if float(np.mean(center_px)) >= float(np.mean(surround_px)) else "black_hot"


def _mask_target_blob(
    gray: np.ndarray,
    valid_region: np.ndarray,
    polarity: str = "white_hot",
    percentile: float = 99.0,
    dilation_px: int = 28,
    min_blob_px: int = 4,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Detecta e mascara o blob de alto contraste (o alvo UAP) na região válida.

    Retorna (mascara_sem_blob, mascara_só_do_blob_dilatado).
    A máscara do blob é retornada separadamente para ser reutilizada pelo
    detector de texto HUD, evitando que o UAP seja re-detectado como 'letra'.
    """
    valid_px = gray[valid_region > 0]
    if valid_px.size < 200:
        return valid_region.copy(), np.zeros_like(valid_region)

    if polarity == "white_hot":
        thresh = max(float(np.percentile(valid_px, percentile)), 170.0)
        candidate = (gray >= thresh).astype(np.uint8) * 255
    else:
        thresh = min(float(np.percentile(valid_px, 100.0 - percentile)), 80.0)
        candidate = (gray <= thresh).astype(np.uint8) * 255

    candidate = cv2.bitwise_and(candidate, valid_region)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(candidate, connectivity=8)

    blob_mask = np.zeros_like(candidate)
    if n_labels > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        if areas.max() >= min_blob_px:
            largest_label = int(np.argmax(areas)) + 1
            blob_mask[labels == largest_label] = 255

    blob_dilated = blob_mask.copy()
    if np.count_nonzero(blob_dilated) > 0 and dilation_px > 0:
        d = dilation_px * 2 + 1
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (d, d))
        blob_dilated = cv2.dilate(blob_dilated, k)

    after_blob = cv2.bitwise_and(valid_region, cv2.bitwise_not(blob_dilated))
    return after_blob, blob_dilated


def _auto_mask_hud_text(
    gray: np.ndarray,
    valid_region: np.ndarray,
    uap_blob_mask: np.ndarray,
    polarity: str = "white_hot",
    text_intensity_percentile: float = 97.5,
    min_text_area_px: int = 8,
    max_text_area_px: int = 800,
    max_aspect_ratio: float = 6.0,
    dilation_px: int = 6,
) -> np.ndarray:
    """
    Detecta e mascara automaticamente texto fixo da HUD (ex: letra 'N' da bússola,
    números de altitude, indicadores de heading, etc.).

    PROBLEMA QUE RESOLVE
    --------------------
    Elementos de texto da HUD são FIXOS na imagem — não se movem com o fundo.
    Se detectados pelo Shi-Tomasi, produzem vetores de deslocamento ≈ 0,
    puxando a mediana para baixo e subestimando o movimento real da câmera.
    O impacto é maior em pans lentos (< 3 px/frame), onde o vetor zero do
    texto pode cair dentro da banda de inliers do filtro MAD.

    ALGORITMO
    ---------
    1. Limiar adaptativo (percentil 97.5 da região válida) → pixels extremos
    2. Exclui o blob do UAP (já identificado) para evitar dupla detecção
    3. Componentes conectados → candidatos a texto
    4. Filtra por tamanho: entre min_text_area_px e max_text_area_px
       - Muito pequeno  → ruído de compressão de vídeo
       - Muito grande   → nuvem brilhante ou artefato, não texto
    5. Filtra por proporção: aspect_ratio <= max_aspect_ratio
       - Filamentos (bordas longas) são rejeitados
    6. Dilata levemente para cobrir antialiasing dos caracteres

    Parâmetros
    ----------
    text_intensity_percentile : percentil de brilho para candidatos a texto.
                                97.5 detecta o 2.5% mais extremo além do UAP.
                                Reduza para 95 se o texto for menos brilhante.
    min_text_area_px          : área mínima de um elemento de texto (pixels)
    max_text_area_px          : área máxima — acima disso não é texto HUD
    max_aspect_ratio          : largura/altura máxima (filtra filamentos)
    dilation_px               : pixels de margem ao redor de cada elemento

    Retorna
    -------
    Máscara com texto HUD removido (válido AND NOT texto_detectado)
    """
    valid_px = gray[valid_region > 0]
    if valid_px.size < 200:
        return valid_region.copy()

    if polarity == "white_hot":
        thresh = float(np.percentile(valid_px, text_intensity_percentile))
        candidate = (gray >= thresh).astype(np.uint8) * 255
    else:
        thresh = float(np.percentile(valid_px, 100.0 - text_intensity_percentile))
        candidate = (gray <= thresh).astype(np.uint8) * 255

    # Restringir à região válida e excluir o blob do UAP
    candidate = cv2.bitwise_and(candidate, valid_region)
    candidate = cv2.bitwise_and(candidate, cv2.bitwise_not(uap_blob_mask))

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(candidate, connectivity=8)

    text_mask = np.zeros_like(candidate)
    for lbl in range(1, n_labels):
        area = int(stats[lbl, cv2.CC_STAT_AREA])
        bw   = int(stats[lbl, cv2.CC_STAT_WIDTH])
        bh   = int(stats[lbl, cv2.CC_STAT_HEIGHT])

        if area < min_text_area_px or area > max_text_area_px:
            continue
        aspect = max(bw, bh) / max(min(bw, bh), 1)
        if aspect > max_aspect_ratio:
            continue

        text_mask[labels == lbl] = 255

    if np.count_nonzero(text_mask) > 0 and dilation_px > 0:
        d = dilation_px * 2 + 1
        k = cv2.getStructuringElement(cv2.MORPH_RECT, (d, d))
        text_mask = cv2.dilate(text_mask, k)

    return cv2.bitwise_and(valid_region, cv2.bitwise_not(text_mask))


def _mask_reticle(
    shape: tuple[int, int],
    center_frac: float = 0.12,
    line_frac: float = 0.015,
) -> np.ndarray:
    """
    Mascara a mira/crosshair central do sistema de armas.
    Remove banda horizontal, banda vertical e caixa central.
    """
    h, w = shape[:2]
    mask = np.ones((h, w), dtype=np.uint8) * 255
    cy, cx = h // 2, w // 2

    lh = max(1, int(h * line_frac / 2))
    mask[max(0, cy - lh) : min(h, cy + lh + 1), :] = 0

    lw = max(1, int(w * line_frac / 2))
    mask[:, max(0, cx - lw) : min(w, cx + lw + 1)] = 0

    hw = max(1, int(w * center_frac / 2))
    hh = max(1, int(h * center_frac / 2))
    mask[
        max(0, cy - hh) : min(h, cy + hh + 1),
        max(0, cx - hw) : min(w, cx + hw + 1),
    ] = 0
    return mask


def _build_background_mask(
    gray: np.ndarray,
    polarity: str = "white_hot",
    black_thresh: int = 18,
    center_frac: float = 0.12,
    line_frac: float = 0.015,
    blob_percentile: float = 99.0,
    blob_dilation_px: int = 28,
    auto_mask_text: bool = True,
    hud_boxes: list[tuple[int, int, int, int]] | None = None,
) -> tuple[np.ndarray, dict[str, float]]:
    """
    Constrói a máscara completa de background em 5 etapas:

      1. Região válida       → remove blocos pretos (overlay do sistema)
      2. Blob do UAP         → remove o alvo térmico de alto contraste
      3. Texto HUD auto      → remove letras/números fixos (ex: 'N', altitude)
      4. Retículo central    → remove crosshair e caixa da mira
      5. Caixas manuais      → exclui regiões adicionais informadas pelo usuário

    O texto HUD (etapa 3) usa a máscara do blob do UAP (etapa 2) para
    não re-detectar o próprio alvo como 'elemento de texto'.
    """
    h, w = gray.shape[:2]

    # 1. Região válida
    valid = _find_valid_region(gray, black_thresh)
    valid_cov = float(np.count_nonzero(valid)) / valid.size

    # 2. Blob do UAP — recebe também a máscara isolada do blob para reuso
    after_blob, uap_blob_mask = _mask_target_blob(
        gray, valid, polarity, blob_percentile, blob_dilation_px
    )

    # 3. Texto HUD automático (usa blob do UAP para evitar dupla detecção)
    if auto_mask_text:
        after_blob = _auto_mask_hud_text(
            gray, after_blob, uap_blob_mask, polarity=polarity
        )

    # 4. Retículo central
    reticle = _mask_reticle(gray.shape, center_frac, line_frac)
    mask = cv2.bitwise_and(after_blob, reticle)

    # 5. Caixas HUD manuais
    for (bx, by, bw, bh) in (hud_boxes or []):
        bx, by, bw, bh = int(bx), int(by), int(bw), int(bh)
        mask[max(0, by) : min(h, by + bh), max(0, bx) : min(w, bx + bw)] = 0

    bg_cov = float(np.count_nonzero(mask)) / mask.size
    return mask, {"valid_coverage": valid_cov, "bg_coverage": bg_cov}


print("Bloco 1 (máscara automática) carregado.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  BLOCO 2 — ESTIMATIVA DE MOVIMENTO POR CORRELAÇÃO DE FASE
# ═══════════════════════════════════════════════════════════════════════════════

def _soft_mask(mask: np.ndarray, blur_px: int = 15) -> np.ndarray:
    """
    Suaviza as bordas da máscara com Gaussiana (soft mask) para uso em FFT.

    Bordas abruptas introduzem descontinuidades que geram componentes de
    alta frequência espúrias no espectro FFT, produzindo picos de correlação
    falsos. A suavização das bordas elimina esses artefatos mantendo a
    exclusão das regiões mascaradas.

    Retorna array float64 com valores em [0.0, 1.0].
    """
    binary = (mask > 0).astype(np.float64)
    if blur_px <= 0:
        return binary
    b = blur_px | 1  # garante kernel ímpar
    return cv2.GaussianBlur(binary, (b, b), blur_px / 3.0)


def _gradient_magnitude(gray: np.ndarray, ksize: int = 5) -> np.ndarray:
    """
    Magnitude do gradiente Sobel — detecta interfaces de contraste.

    Em vídeos FLIR o background tem textura muito suave (nuvens, névoa,
    terreno distante). Cantos Shi-Tomasi são raros e instáveis nesses frames.
    Mas o background possui INTERFACES DE CONTRASTE AMPLAS E ESTÁVEIS:
      • Linha do horizonte  (transição abrupta céu/terra)
      • Bordas de camadas de nuvens
      • Gradientes térmicos horizontais no terreno

    O gradiente Sobel destaca exatamente essas interfaces, que servem como
    'âncoras' para a correlação de fase — muito mais confiáveis que cantos
    em texturas homogêneas.

    ksize=5 captura tanto transições finas (1-2 px) quanto suaves (bordas
    difusas de nuvens), que são o padrão dominante em vídeos FLIR de altitude.
    """
    g = gray.astype(np.float64)
    gx = cv2.Sobel(g, cv2.CV_64F, 1, 0, ksize=ksize)
    gy = cv2.Sobel(g, cv2.CV_64F, 0, 1, ksize=ksize)
    return np.sqrt(gx * gx + gy * gy)


def _phase_shift(
    img1: np.ndarray,
    img2: np.ndarray,
    hann: np.ndarray | None = None,
) -> tuple[float, float, float]:
    """
    Deslocamento sub-pixel (dx, dy) entre dois frames via correlação de fase (FFT).

    A correlação de fase calcula o espectro cruzado normalizado:
      C(f) = FFT(img1) × conj(FFT(img2)) / |FFT(img1) × conj(FFT(img2))|
    O pico de IFFT(C) corresponde ao deslocamento de translação dominante.

    Diferente de métodos pontuais (Lucas-Kanade), opera sobre TODA A ÁREA
    DE BACKGROUND DE UMA VEZ — cada pixel de interface de contraste contribui
    para o voto do deslocamento dominante.

    Parâmetros
    ----------
    img1, img2 : imagens float64 de mesmo tamanho (mascaradas)
    hann       : janela de Hanning (suprime periodicidade de borda do FFT)

    Retorna
    -------
    (dx, dy, response)
    dx, dy   : deslocamento sub-pixel em pixels
    response : altura normalizada do pico ∈ [0, 1]
               < 0.02 → background muito homogêneo (pouca estrutura rastreável)
    """
    try:
        i1 = np.ascontiguousarray(img1, dtype=np.float64)
        i2 = np.ascontiguousarray(img2, dtype=np.float64)
        shift, response = cv2.phaseCorrelate(i1, i2, hann)
        return float(shift[0]), float(shift[1]), float(response)
    except cv2.error:
        return 0.0, 0.0, 0.0


def _estimate_frame_motion(
    prev_gray: np.ndarray,
    curr_gray: np.ndarray,
    bg_mask: np.ndarray,
    soft_blur_px: int = 15,
    min_response: float = 0.02,
    min_coverage: float = 0.03,
) -> dict[str, Any]:
    """
    Estima bg_dx, bg_dy por correlação de fase sobre o gradiente do background.

    Pipeline
    ────────
    1. soft_mask     → suaviza bordas da máscara (evita artefatos FFT)
    2. gradiente     → magnitude Sobel (interfaces de contraste)
    3. × soft_mask   → exclui UAP, mira, HUD e bordas pretas
    4. Hanning       → janela espectral (suprime periodicidade de borda)
    5. phaseCorrelate → pico = deslocamento dominante com precisão sub-pixel

    Retorna
    ───────
    dict com:
      bg_dx_px, bg_dy_px : deslocamento estimado do fundo em pixels
      n_features         : pixels de background disponíveis (cobertura)
      n_inliers          : igual a n_features se ok, 0 caso contrário
      confidence         : resposta da correlação de fase ∈ [0, 1]
      motion_ok          : True se a resposta superou min_response
    """
    n_bg = int(np.count_nonzero(bg_mask))
    result: dict[str, Any] = {
        "bg_dx_px": 0.0, "bg_dy_px": 0.0,
        "n_features": n_bg, "n_inliers": 0,
        "confidence": 0.0, "motion_ok": False,
    }

    if n_bg < int(bg_mask.size * min_coverage):
        return result

    h, w = prev_gray.shape[:2]

    # 1. Soft mask — bordas suaves para FFT
    soft = _soft_mask(bg_mask, soft_blur_px)

    # 2+3. Gradiente × soft mask — interfaces de contraste excluindo UAP/HUD
    g1 = _gradient_magnitude(prev_gray) * soft
    g2 = _gradient_magnitude(curr_gray) * soft

    # Verificação: gradiente quase nulo = background completamente liso
    if g1.max() < 1e-6 or g2.max() < 1e-6:
        return result

    # 4. Janela de Hanning — (largura × altura) — convenção OpenCV cols × rows
    hann = cv2.createHanningWindow((w, h), cv2.CV_64F)

    # 5. Correlação de fase → deslocamento dominante sub-pixel
    dx, dy, response = _phase_shift(g1, g2, hann)

    ok = response >= min_response
    result.update({
        "bg_dx_px": dx if ok else 0.0,
        "bg_dy_px": dy if ok else 0.0,
        "n_inliers": n_bg if ok else 0,
        "confidence": response,
        "motion_ok": ok,
    })
    return result


print("Bloco 2 (correlação de fase sobre gradiente) carregado.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  BLOCO 3 — PROCESSAMENTO DE SINAL: POSIÇÃO, VELOCIDADE E ACELERAÇÃO
# ═══════════════════════════════════════════════════════════════════════════════

def _prefilter_displacements(
    dx: np.ndarray,
    dy: np.ndarray,
    ok: np.ndarray,
    window: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Pré-filtra os deslocamentos brutos frame-a-frame com mediana móvel centrada.

    Frames onde a estimativa falhou (ok=False) são marcados como NaN e
    interpolados pelo valor mediano dos vizinhos. Isso evita que frames
    problemáticos (cena totalmente desfocada, pan muito rápido, background
    quase liso) injetem saltos espúrios na integral de posição.

    O window deve ser ímpar para simetria. A mediana (vs média) mantém
    robustez a outliers que passaram pelo filtro MAD do LK.

    Parâmetros
    ----------
    dx, dy : deslocamentos brutos por frame (background)
    ok     : bool array — True onde a estimativa foi bem-sucedida
    window : janela da mediana móvel (frames)

    Retorna
    -------
    (dx_filtrado, dy_filtrado) como numpy arrays
    """
    w = window if window % 2 == 1 else window + 1
    w = max(w, 3)

    s_dx = pd.Series(np.where(ok, dx, np.nan))
    s_dy = pd.Series(np.where(ok, dy, np.nan))

    dx_filt = (
        s_dx.rolling(w, center=True, min_periods=1)
        .median()
        .fillna(0)
        .to_numpy(dtype=float)
    )
    dy_filt = (
        s_dy.rolling(w, center=True, min_periods=1)
        .median()
        .fillna(0)
        .to_numpy(dtype=float)
    )
    return dx_filt, dy_filt


def _compute_kinematics(
    position: np.ndarray,
    dt: float,
    window: int,
    poly: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Calcula posição suavizada, velocidade e aceleração via filtro Savitzky-Golay.

    O SG ajusta um polinômio local de grau `poly` a uma janela de `window`
    pontos. As derivadas são calculadas ANALITICAMENTE nesse polinômio —
    não numericamente — o que evita a amplificação de ruído que ocorre ao
    diferenciar numericamente depois de suavizar.

    Resultado:
      deriv=0 → posição suavizada
      deriv=1 → velocidade em px/s (divide pelo dt automaticamente)
      deriv=2 → aceleração em px/s² (divide por dt² automaticamente)

    Se scipy não estiver disponível, usa numpy.gradient (mais ruidoso).
    """
    if not _SCIPY or len(position) < window:
        smooth = position.copy()
        vel = np.gradient(smooth, dt)
        acc = np.gradient(vel, dt)
        return smooth, vel, acc

    smooth = savgol_filter(position, window, poly)
    vel    = savgol_filter(position, window, poly, deriv=1, delta=dt)
    acc    = savgol_filter(position, window, poly, deriv=2, delta=dt)
    return smooth, vel, acc


def _make_sg_window(n: int, window_s: float, eff_fps: float, poly: int) -> int:
    """
    Calcula tamanho de janela válido para Savitzky-Golay.

    Restrições:
      - Deve ser ímpar (requisito do SG)
      - Deve ser > poly (requisito do SG)
      - Deve ser <= número de amostras
    """
    w = max(poly + 2, int(round(window_s * eff_fps)))
    if w % 2 == 0:
        w += 1
    max_w = n if n % 2 != 0 else n - 1
    w = min(w, max(poly + 1, max_w))
    # Garantir paridade ímpar após clamp
    if w % 2 == 0:
        w -= 1
    return max(w, poly + 1 if (poly + 1) % 2 != 0 else poly + 2)


print("Bloco 3 (processamento de sinal) carregado.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  BLOCO 4 — FUNÇÃO PRINCIPAL
# ═══════════════════════════════════════════════════════════════════════════════

def _safe_fps(cap: cv2.VideoCapture, fallback: float = 30.0) -> float:
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    return fps if (math.isfinite(fps) and fps > 0) else fallback


def estimate_camera_motion(
    video_path: str | Path,
    *,
    # ── Detecção da região válida (blocos pretos) ──────────────────────────
    black_thresh: int = 18,
    # ── Polaridade térmica (white-hot vs black-hot) ────────────────────────
    target_polarity: str = "auto",
    # ── Mascaramento do blob do alvo (UAP) ────────────────────────────────
    blob_percentile: float = 99.0,
    blob_dilation_px: int = 28,
    # ── Mascaramento automático de texto HUD ──────────────────────────────
    auto_mask_text: bool = True,
    # ── Mascaramento do retículo/mira ──────────────────────────────────────
    center_frac: float = 0.12,
    line_frac: float = 0.015,
    # ── Caixas HUD adicionais (texto, indicadores fixos) ───────────────────
    hud_boxes: list[tuple[int, int, int, int]] | None = None,
    # ── Correlação de fase ─────────────────────────────────────────────────
    soft_blur_px: int = 15,
    min_response: float = 0.02,
    # ── Seleção de frames ──────────────────────────────────────────────────
    frame_step: int = 1,
    roi: tuple[int, int, int, int] | None = None,
    fps_override: float | None = None,
    # ── Pré-filtro de deslocamentos brutos ────────────────────────────────
    prefilter_window: int = 5,
    # ── Suavização Savitzky-Golay (posição → velocidade → aceleração) ──────
    smooth_window_s: float = 0.5,
    smooth_poly: int = 3,
    # ── Misc ───────────────────────────────────────────────────────────────
    verbose: bool = True,
) -> dict[str, Any]:
    """
    Estima posição, velocidade e aceleração da câmera FLIR de sistema de armas.

    A estimativa é feita EXCLUSIVAMENTE pelo movimento do background, usando
    correlação de fase sobre a magnitude do gradiente (interfaces de contraste).
    Todos os elementos fixos ou de alto contraste são mascarados automaticamente.

    Método de estimativa
    --------------------
    Usa correlação de fase (FFT cross-power spectrum) sobre a imagem de gradiente
    Sobel do background mascarado. O pico da correlação cruzada normalizada dá
    o deslocamento dominante da cena com precisão sub-pixel.

    Por que gradiente + correlação de fase?
    - Background FLIR tem textura suave → poucos cantos Shi-Tomasi (LK falha)
    - Mas possui interfaces de contraste amplas (horizonte, bordas de nuvens)
    - O gradiente destaca essas interfaces; a correlação de fase as alinha
    - Opera sobre TODA a área de background — não sobre pontos esparsos

    Parâmetros de mascaramento
    --------------------------
    black_thresh      : limiar para pixels pretos do overlay HUD (padrão: 18)
    target_polarity   : 'auto' | 'white_hot' | 'black_hot'
    blob_percentile   : percentil para detectar o UAP (padrão: 99)
    blob_dilation_px  : raio de dilatação do blob do UAP em pixels
    auto_mask_text    : True = detecta e mascara texto HUD automaticamente
    center_frac       : tamanho da caixa central da mira (fração do frame)
    hud_boxes         : [(x, y, w, h)] de regiões HUD adicionais a excluir

    Parâmetros de correlação de fase
    ---------------------------------
    soft_blur_px : raio do blur Gaussiano nas bordas da máscara (evita artefatos FFT)
    min_response : resposta mínima do pico de correlação (0.02 = 2% do máximo)
                   Frames abaixo disso têm background sem estrutura rastreável.
                   Reduza para 0.01 se o vídeo tiver background muito liso.

    Parâmetros de sinal
    --------------------
    smooth_window_s : janela do filtro Savitzky-Golay em segundos
    smooth_poly     : grau do polinômio SG (3 = padrão)

    Retorna
    -------
    dict com 'metadata' e 'data' (pd.DataFrame)

    Colunas principais do DataFrame
    --------------------------------
    frame_index, time_s
    bg_dx_px, bg_dy_px           : deslocamento bruto do fundo por frame
    bg_dx_filt_px, bg_dy_filt_px : deslocamento pré-filtrado (mediana móvel)
    cam_pos_x_px, cam_pos_y_px   : posição cumulativa da câmera (= -sum bg)
    cam_pos_x_smooth_px, [...]   : posição suavizada pelo SG
    cam_vel_x_px_s, cam_vel_y_px_s    : velocidade em px/s
    cam_acc_x_px_s2, cam_acc_y_px_s2  : aceleração em px/s²
    cam_speed_px_s, cam_accel_magnitude_px_s2
    confidence  : resposta da correlação de fase ∈ [0, 1]
    motion_ok   : True se a estimativa foi bem-sucedida
    """
    video_path = Path(video_path).expanduser().resolve()
    if not video_path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {video_path}")
    if frame_step < 1:
        raise ValueError("frame_step deve ser >= 1")
    if target_polarity not in {"auto", "white_hot", "black_hot"}:
        raise ValueError("target_polarity: 'auto' | 'white_hot' | 'black_hot'")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Não foi possível abrir o vídeo: {video_path}")

    fps   = fps_override or _safe_fps(cap)
    n_nom = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    vid_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)  or 0)
    vid_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)

    if verbose:
        print(f"Vídeo  : {video_path.name}")
        print(f"         {vid_w}×{vid_h} px  |  {fps:.3f} fps  |  ~{n_nom} frames")

    if roi is not None:
        rx, ry, rw, rh = (int(v) for v in roi)
        if not (0 <= rx and 0 <= ry and rw > 0 and rh > 0
                and rx + rw <= vid_w and ry + rh <= vid_h):
            raise ValueError(f"roi {roi} inválido para frame {vid_w}×{vid_h}")
    else:
        rx, ry, rw, rh = 0, 0, vid_w, vid_h

    dt      = frame_step / fps
    eff_fps = fps / frame_step

    rows: list[dict[str, Any]] = []
    prev_gray: np.ndarray | None = None
    polarity  = target_polarity
    read_idx  = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        fi = read_idx
        read_idx += 1

        if fi % frame_step != 0:
            continue

        t_s  = fi / fps
        crop = frame[ry : ry + rh, rx : rx + rw]
        gray = _to_gray(crop)

        # Primeiro frame: detecta polaridade
        if prev_gray is None:
            if target_polarity == "auto":
                valid0   = _find_valid_region(gray, black_thresh)
                polarity = _detect_thermal_polarity(gray, valid0)
                if verbose:
                    print(f"Polaridade detectada : {polarity}")
            rows.append({
                "frame_index": fi, "time_s": t_s,
                "bg_dx_px": 0.0, "bg_dy_px": 0.0,
                "n_features": 0, "n_inliers": 0,
                "confidence": 0.0,
                "bg_coverage": np.nan, "valid_coverage": np.nan,
                "motion_ok": False,
            })
            prev_gray = gray
            continue

        # Máscara de background
        bg_mask, cov = _build_background_mask(
            gray,
            polarity=polarity,
            black_thresh=black_thresh,
            center_frac=center_frac,
            line_frac=line_frac,
            blob_percentile=blob_percentile,
            blob_dilation_px=blob_dilation_px,
            auto_mask_text=auto_mask_text,
            hud_boxes=hud_boxes,
        )

        # Estimativa de movimento por correlação de fase
        motion = _estimate_frame_motion(
            prev_gray, gray, bg_mask,
            soft_blur_px=soft_blur_px,
            min_response=min_response,
        )

        rows.append({
            "frame_index": fi, "time_s": t_s,
            **motion,
            **cov,
        })
        prev_gray = gray

    cap.release()

    if not rows:
        raise RuntimeError("Nenhum frame foi processado.")

    df = pd.DataFrame(rows)
    n  = len(df)

    # ── Pré-filtrar deslocamentos brutos ─────────────────────────────────
    dx_raw = df["bg_dx_px"].to_numpy(dtype=float)
    dy_raw = df["bg_dy_px"].to_numpy(dtype=float)
    ok_arr = df["motion_ok"].to_numpy(dtype=bool)

    dx_filt, dy_filt = _prefilter_displacements(
        dx_raw, dy_raw, ok_arr, prefilter_window
    )
    df["bg_dx_filt_px"] = dx_filt
    df["bg_dy_filt_px"] = dy_filt

    # ── Posição cumulativa da câmera ─────────────────────────────────────
    df["cam_pos_x_px"] = -np.cumsum(dx_filt)
    df["cam_pos_y_px"] = -np.cumsum(dy_filt)

    # ── Savitzky-Golay: posição suave, velocidade, aceleração ────────────
    sg_w = _make_sg_window(n, smooth_window_s, eff_fps, smooth_poly)

    if verbose:
        n_ok = int(ok_arr.sum())
        print(f"\nFrames processados   : {n}")
        print(f"Janela SG            : {sg_w} frames ({sg_w * dt:.3f} s)")
        print(f"Estimativas OK       : {n_ok}/{max(n-1,1)} ({100*n_ok/max(n-1,1):.1f}%)")
        mc = df["bg_coverage"].dropna()
        if not mc.empty:
            print(f"Cobertura BG (média) : {mc.mean():.1%}")
        cr = df["confidence"].dropna()
        if not cr.empty:
            print(f"Resposta correlação  : {cr.mean():.4f} (média) | "
                  f"{cr.max():.4f} (max)")

    px = df["cam_pos_x_px"].to_numpy(dtype=float)
    py = df["cam_pos_y_px"].to_numpy(dtype=float)

    sx, vx, ax_ = _compute_kinematics(px, dt, sg_w, smooth_poly)
    sy, vy, ay_ = _compute_kinematics(py, dt, sg_w, smooth_poly)

    df["cam_pos_x_smooth_px"]       = sx
    df["cam_pos_y_smooth_px"]       = sy
    df["cam_vel_x_px_s"]            = vx
    df["cam_vel_y_px_s"]            = vy
    df["cam_acc_x_px_s2"]           = ax_
    df["cam_acc_y_px_s2"]           = ay_
    df["cam_speed_px_s"]            = np.hypot(vx, vy)
    df["cam_accel_magnitude_px_s2"] = np.hypot(ax_, ay_)

    metadata: dict[str, Any] = {
        "video_path"       : str(video_path),
        "fps"              : fps,
        "frame_step"       : frame_step,
        "eff_fps"          : eff_fps,
        "dt_s"             : dt,
        "video_size"       : (vid_w, vid_h),
        "roi"              : (rx, ry, rw, rh),
        "thermal_polarity" : polarity,
        "black_thresh"     : black_thresh,
        "blob_percentile"  : blob_percentile,
        "blob_dilation_px" : blob_dilation_px,
        "auto_mask_text"   : auto_mask_text,
        "center_frac"      : center_frac,
        "line_frac"        : line_frac,
        "soft_blur_px"     : soft_blur_px,
        "min_response"     : min_response,
        "smooth_window_s"  : smooth_window_s,
        "smooth_poly"      : smooth_poly,
        "sg_window_frames" : sg_w,
        "n_frames"         : n,
        "n_ok"             : int(ok_arr.sum()),
        "hud_boxes"        : hud_boxes or [],
    }

    return {"metadata": metadata, "data": df}


print("Bloco 4 (função principal) carregado.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  BLOCO 5 — VISUALIZAÇÃO
# ═══════════════════════════════════════════════════════════════════════════════

def _shade_bad_frames(
    ax: plt.Axes,
    times: np.ndarray,
    ok: np.ndarray,
    alpha: float = 0.15,
) -> None:
    """Sombreia em vermelho claro os intervalos onde motion_ok=False."""
    bad = ~ok.astype(bool)
    if not bad.any():
        return
    in_bad = False
    t_start = None
    for t, b in zip(times, bad):
        if b and not in_bad:
            t_start = t
            in_bad = True
        elif not b and in_bad:
            ax.axvspan(t_start, t, color="red", alpha=alpha, linewidth=0)
            in_bad = False
    if in_bad:
        ax.axvspan(t_start, times[-1], color="red", alpha=alpha, linewidth=0)


def plot_camera_kinematics(
    result: dict[str, Any],
    figsize: tuple[float, float] = (15, 11),
    title: str | None = None,
    shade_failures: bool = True,
) -> plt.Figure:
    """
    Gráfico principal: posição, velocidade e aceleração da câmera em X e Y.

    Layout 3×2: Posição X/Y | Velocidade X/Y | Aceleração X/Y
    Regiões em vermelho = frames onde a correlação de fase não encontrou
    estrutura suficiente no background para estimar o movimento.
    """
    df   = result["data"]
    meta = result["metadata"]
    t    = df["time_s"].to_numpy()
    ok   = df["motion_ok"].to_numpy()

    fig, axes = plt.subplots(3, 2, figsize=figsize, sharex=True)
    _t = title or Path(meta["video_path"]).name
    fig.suptitle(f"Cinemática da Câmera FLIR\n{_t}", fontsize=13, fontweight="bold")

    specs = [
        ("cam_pos_x_px", "cam_pos_x_smooth_px", "cam_vel_x_px_s",
         "cam_acc_x_px_s2", "#2166ac", "X", "Pan   (+= direita)",  "Vel X", "Acel X"),
        ("cam_pos_y_px", "cam_pos_y_smooth_px", "cam_vel_y_px_s",
         "cam_acc_y_px_s2", "#4dac26", "Y", "Tilt  (+= baixo)",   "Vel Y", "Acel Y"),
    ]
    vel_colors = ["#d6604d", "#f4a582"]
    acc_colors = ["#762a83", "#af8dc3"]

    for col_idx, (raw, smooth, vel, acc, color, ax_lbl,
                  pos_title, vel_title, acc_title) in enumerate(specs):

        ax = axes[0, col_idx]
        ax.plot(t, df[raw],    color=color, alpha=0.2, lw=1,   label="raw")
        ax.plot(t, df[smooth], color=color, lw=2.2,            label="SG suavizado")
        ax.axhline(0, color="k", lw=0.7, alpha=0.3)
        ax.set_ylabel(f"Posição {ax_lbl} (px)", fontsize=10)
        ax.set_title(pos_title, fontsize=9)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.25)
        if shade_failures: _shade_bad_frames(ax, t, ok)

        ax = axes[1, col_idx]
        ax.plot(t, df[vel], color=vel_colors[col_idx], lw=1.8)
        ax.axhline(0, color="k", lw=0.7, alpha=0.3)
        ax.set_ylabel(f"Velocidade {ax_lbl} (px/s)", fontsize=10)
        ax.set_title(vel_title, fontsize=9)
        ax.grid(True, alpha=0.25)
        if shade_failures: _shade_bad_frames(ax, t, ok)

        ax = axes[2, col_idx]
        ax.plot(t, df[acc], color=acc_colors[col_idx], lw=1.8)
        ax.axhline(0, color="k", lw=0.7, alpha=0.3)
        ax.set_ylabel(f"Aceleração {ax_lbl} (px/s²)", fontsize=10)
        ax.set_xlabel("Tempo (s)", fontsize=10)
        ax.set_title(acc_title, fontsize=9)
        ax.grid(True, alpha=0.25)
        if shade_failures: _shade_bad_frames(ax, t, ok)

    sg_w = meta["sg_window_frames"]
    footer = (
        f"Polaridade: {meta['thermal_polarity']}  |  "
        f"Correlação de fase  |  soft_blur={meta['soft_blur_px']} px  |  "
        f"min_response={meta['min_response']}  |  "
        f"SG: {sg_w} frames ({meta['smooth_window_s']:.2f}s, poly={meta['smooth_poly']})  |  "
        f"OK: {meta['n_ok']}/{meta['n_frames']-1}"
    )
    fig.text(0.5, 0.0, footer, ha="center", fontsize=7.5, color="#555555")
    fig.tight_layout(rect=[0, 0.02, 1, 1])
    return fig


def plot_camera_trajectory(
    result: dict[str, Any],
    figsize: tuple[float, float] = (7, 6),
) -> plt.Figure:
    """Trajetória 2D da câmera no plano da imagem, colorida por tempo (paleta plasma)."""
    df   = result["data"]
    meta = result["metadata"]
    x    = df["cam_pos_x_smooth_px"].to_numpy()
    y    = df["cam_pos_y_smooth_px"].to_numpy()
    t    = df["time_s"].to_numpy()

    fig, ax = plt.subplots(figsize=figsize)
    pts  = np.stack([x, y], axis=1).reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc   = LineCollection(segs, cmap="plasma", linewidth=2.2,
                          norm=Normalize(t.min(), t.max()))
    lc.set_array(t[:-1])
    ax.add_collection(lc)
    cb = fig.colorbar(lc, ax=ax, pad=0.02)
    cb.set_label("Tempo (s)", fontsize=10)
    ax.scatter(x[0],  y[0],  c="royalblue", s=90, zorder=5, label="Início", marker="o")
    ax.scatter(x[-1], y[-1], c="crimson",   s=100, zorder=5, label="Fim",    marker="*")
    ax.set_xlabel("Câmera X (px)  [+ = pan direita]", fontsize=10)
    ax.set_ylabel("Câmera Y (px)  [+ = tilt baixo]",  fontsize=10)
    ax.set_title(f"Trajetória 2D da Câmera\n{Path(meta['video_path']).name}", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)
    ax.autoscale()
    ax.set_aspect("equal", adjustable="datalim")
    fig.tight_layout()
    return fig


def plot_motion_quality(
    result: dict[str, Any],
    figsize: tuple[float, float] = (14, 5),
) -> plt.Figure:
    """
    Métricas de qualidade da correlação de fase frame-a-frame:
      1. Deslocamento bruto por frame (bg_dx e bg_dy)
      2. Resposta da correlação de fase (confiança do pico FFT)
      3. Cobertura do background (fração do frame usada)

    Use este gráfico para ajustar min_response, blob_dilation_px e hud_boxes.
    """
    df = result["data"]
    t  = df["time_s"].to_numpy()

    fig, axes = plt.subplots(1, 3, figsize=figsize)
    fig.suptitle(
        "Qualidade da Estimativa — Correlação de Fase sobre Gradiente",
        fontsize=12, fontweight="bold",
    )

    # Painel 1: deslocamento bruto por frame
    ax = axes[0]
    ax.plot(t, df["bg_dx_px"], color="#2166ac", lw=1.2, alpha=0.8, label="dx (horizontal)")
    ax.plot(t, df["bg_dy_px"], color="#4dac26", lw=1.2, alpha=0.8, label="dy (vertical)")
    ax.axhline(0, color="k", lw=0.7, alpha=0.3)
    ax.set_xlabel("Tempo (s)")
    ax.set_ylabel("Deslocamento (px/frame)")
    ax.set_title("Deslocamento bruto do background")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

    # Painel 2: resposta da correlação de fase
    ax = axes[1]
    ax.plot(t, df["confidence"], color="#d6604d", lw=1.5)
    min_r = result["metadata"].get("min_response", 0.02)
    ax.axhline(min_r, color="crimson", ls="--", lw=1,
               label=f"limiar min_response={min_r}")
    ax.set_xlabel("Tempo (s)")
    ax.set_ylabel("Resposta do pico FFT")
    ax.set_title("Confiança da correlação de fase")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

    # Painel 3: cobertura do background
    ax = axes[2]
    ax.plot(t, df["bg_coverage"] * 100, color="#8c510a", lw=1.5)
    ax.axhline(3, color="crimson", ls="--", lw=1, label="mínimo útil (3%)")
    ax.set_xlabel("Tempo (s)")
    ax.set_ylabel("Cobertura BG (%)")
    ax.set_title("Fração do frame usada como background")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

    fig.tight_layout()
    return fig


def visualize_mask(
    video_path: str | Path,
    frame_idx: int = 0,
    *,
    black_thresh: int = 18,
    target_polarity: str = "auto",
    blob_percentile: float = 99.0,
    blob_dilation_px: int = 28,
    auto_mask_text: bool = True,
    center_frac: float = 0.12,
    line_frac: float = 0.015,
    hud_boxes: list[tuple[int, int, int, int]] | None = None,
    roi: tuple[int, int, int, int] | None = None,
    soft_blur_px: int = 15,
    figsize: tuple[float, float] = (14, 5),
) -> None:
    """
    Visualiza a máscara de background e as interfaces de contraste usadas
    pela correlação de fase.

    Painel esquerdo : frame original
    Painel direito  : overlay com:
      - Vermelho        = regiões EXCLUÍDAS (UAP, mira, HUD, bordas pretas)
      - Verde brilhante = gradiente forte → interfaces de contraste ATIVAS
                          (o que a correlação de fase vai rastrear)
      - Cinza/escuro    = background válido mas com pouco gradiente
                          (contribui menos para a estimativa de movimento)

    Use esta visualização para verificar:
      1. As regiões excluídas (vermelho) estão corretas?
      2. Há suficiente gradiente verde no background para rastrear?
      3. Nenhuma interface importante está sendo mascarada?
    """
    cap = cv2.VideoCapture(str(Path(video_path).expanduser().resolve()))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if not ret:
        raise ValueError(f"Não foi possível ler o frame {frame_idx}")

    if roi is not None:
        rx, ry, rw, rh = roi
        frame = frame[ry : ry + rh, rx : rx + rw]

    gray = _to_gray(frame)

    polarity = target_polarity
    if polarity == "auto":
        v0 = _find_valid_region(gray, black_thresh)
        polarity = _detect_thermal_polarity(gray, v0)

    mask, cov = _build_background_mask(
        gray, polarity=polarity,
        black_thresh=black_thresh,
        center_frac=center_frac, line_frac=line_frac,
        blob_percentile=blob_percentile,
        blob_dilation_px=blob_dilation_px,
        auto_mask_text=auto_mask_text,
        hud_boxes=hud_boxes,
    )

    # Gradiente mascarado — o que a correlação de fase efetivamente usa
    soft = _soft_mask(mask, soft_blur_px)
    grad = _gradient_magnitude(gray) * soft

    # Normaliza gradiente para [0, 255]
    g_max = float(grad.max())
    if g_max > 1e-6:
        grad_vis = (grad / g_max * 255).clip(0, 255).astype(np.uint8)
    else:
        grad_vis = np.zeros_like(grad, dtype=np.uint8)

    # Monta o overlay
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if frame.ndim == 3 else \
                cv2.cvtColor(frame, cv2.COLOR_GRAY2RGB)

    overlay = frame_rgb.copy().astype(np.float32)
    excluded = mask == 0
    in_bg    = mask > 0

    # Regiões excluídas: tint vermelho
    overlay[excluded, 0] = np.clip(overlay[excluded, 0] * 0.3 + 200, 0, 255)
    overlay[excluded, 1] = overlay[excluded, 1] * 0.3
    overlay[excluded, 2] = overlay[excluded, 2] * 0.3

    # Background ativo: boost no canal verde proporcional ao gradiente
    g_boost = grad_vis[in_bg].astype(float) * 0.7
    overlay[in_bg, 0] = np.clip(overlay[in_bg, 0] - g_boost * 0.3, 0, 255)
    overlay[in_bg, 1] = np.clip(overlay[in_bg, 1] + g_boost,        0, 255)
    overlay[in_bg, 2] = np.clip(overlay[in_bg, 2] - g_boost * 0.3, 0, 255)

    overlay = overlay.astype(np.uint8)

    # Estatísticas do gradiente no background válido
    grad_mean = float(grad[in_bg].mean()) if in_bg.any() else 0.0
    n_strong  = int((grad_vis[in_bg] > 50).sum()) if in_bg.any() else 0

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    axes[0].imshow(frame_rgb)
    axes[0].set_title(f"Frame original (índice {frame_idx})", fontsize=11)
    axes[0].axis("off")

    axes[1].imshow(overlay)
    axes[1].set_title(
        f"Máscara  ·  vermelho=excluído  ·  verde=gradiente ativo (correlação de fase)\n"
        f"Polaridade: {polarity}  |  BG: {cov['bg_coverage']:.1%}  |  "
        f"Válido: {cov['valid_coverage']:.1%}  |  "
        f"Gradiente médio BG: {grad_mean:.1f}  |  "
        f"Pixels ativos (grad>50): {n_strong:,}",
        fontsize=9,
    )
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()


print("Bloco 5 (visualização) carregado.")

## Como usar

### Passo 1 — Verificar a máscara antes de rodar (recomendado)

```python
visualize_mask(
    "meu_video.mp4",
    frame_idx=30,              # escolha um frame representativo
    blob_dilation_px=30,       # aumente se o halo do UAP vazar
    center_frac=0.15,          # aumente se a mira for grande
    hud_boxes=[(580, 10, 80, 25)],  # exemplo: exclui texto HUD no topo
)
```

### Passo 2 — Rodar a análise

```python
result = estimate_camera_motion(
    "meu_video.mp4",
    blob_dilation_px=30,
    center_frac=0.15,
    hud_boxes=[(580, 10, 80, 25)],
)
```

### Passo 3 — Plotar

```python
plot_camera_kinematics(result)   # posição, velocidade, aceleração
plot_camera_trajectory(result)   # trajetória 2D
plot_motion_quality(result)      # métricas de confiabilidade
```

---

### Guia de ajuste de parâmetros

| Sintoma | Diagnóstico | Parâmetro a ajustar |
|---|---|---|
| Cobertura BG < 3% | Máscara removeu quase tudo | Reduza `blob_dilation_px` ou `center_frac` |
| Muitos frames OK=False | Features insuficientes | Reduza `lk_win_size` ou `max_features ↑` |
| Saltos bruscos na posição | Outliers passando pelo MAD | Aumente `blob_dilation_px` |
| Curvas muito ruidosas | Suavização insuficiente | Aumente `smooth_window_s` |
| Manobras não aparecem | Suavização excessiva | Reduza `smooth_window_s` |
| Alvo escuro (black-hot) | Auto-detecção errada | Defina `target_polarity='black_hot'` |
| Blocos pretos não mascarados | Limiar muito baixo | Aumente `black_thresh` (ex: 25) |
| Pan muito rápido | LK perde rastreamento | Aumente `lk_win_size` (ex: 31) e `pyr_levels` |

### Interpretação dos gráficos

- **Posição**: deslocamento cumulativo da câmera desde o início do vídeo. Uma rampa constante = pan uniforme. Platô = câmera parada.
- **Velocidade**: taxa de pan/tilt instantânea. Picos = manobras. Deve ser zero quando câmera está estacionária.
- **Aceleração**: mudança de velocidade. Picos altos = manobras abruptas do gimbal para manter o UAP na mira.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  EXEMPLO DE USO
# ═══════════════════════════════════════════════════════════════════════════════

VIDEO = "/content/seu_video_flir.mp4"   # ← altere para o caminho do vídeo

# ── Passo 1: Verificar máscara e gradiente ────────────────────────────────────
# Verde brilhante = interfaces de contraste ativas (o que será rastreado)
# Vermelho        = regiões excluídas (UAP, mira, HUD, bordas pretas)
visualize_mask(
    VIDEO,
    frame_idx=30,
    black_thresh=18,
    target_polarity="auto",
    blob_percentile=99.0,
    blob_dilation_px=28,
    auto_mask_text=True,
    center_frac=0.12,
    line_frac=0.015,
    soft_blur_px=15,
    hud_boxes=None,     # ex: [(x, y, w, h)] para excluir texto HUD adicional
)

In [ ]:
# ── Passo 2: Rodar estimativa de movimento ────────────────────────────────────
result = estimate_camera_motion(
    VIDEO,

    # Mascaramento
    black_thresh=18,
    target_polarity="auto",
    blob_percentile=99.0,
    blob_dilation_px=28,
    auto_mask_text=True,
    center_frac=0.12,
    line_frac=0.015,
    hud_boxes=None,

    # Correlação de fase
    soft_blur_px=15,       # blur das bordas da máscara (evita artefatos FFT)
    min_response=0.02,     # pico mínimo da correlação (reduza para 0.01 se BG liso)

    # Processamento de sinal
    prefilter_window=5,
    smooth_window_s=0.5,
    smooth_poly=3,

    frame_step=1,
    verbose=True,
)

# Resumo estatístico
df = result["data"]
print("\n── Estatísticas cinemáticas ──")
display(
    df[["cam_vel_x_px_s", "cam_vel_y_px_s",
        "cam_acc_x_px_s2", "cam_acc_y_px_s2",
        "cam_speed_px_s", "cam_accel_magnitude_px_s2"]]
    .describe()
    .round(2)
)

In [ ]:
# ── Passo 3: Gráficos ─────────────────────────────────────────────────────────

# Posição, velocidade e aceleração (X e Y)
fig1 = plot_camera_kinematics(result, shade_failures=True)
plt.show()

# Trajetória 2D colorida por tempo
fig2 = plot_camera_trajectory(result)
plt.show()

# Métricas de confiabilidade do rastreamento
fig3 = plot_motion_quality(result)
plt.show()

In [ ]:
# ── Exportar dados (opcional) ─────────────────────────────────────────────────
# result["data"].to_csv("camera_motion.csv", index=False)
# print("Exportado para camera_motion.csv")

# ── Inspecionar todas as colunas disponíveis ──────────────────────────────────
print("Colunas do DataFrame resultado:")
for col in result["data"].columns:
    sample = result["data"][col].iloc[1] if len(result["data"]) > 1 else None
    print(f"  {col:<35} ex: {sample}")